In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

print("PyTorch version:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

PyTorch version: 2.15.0+cpu
Device: cpu


In [2]:
df = pd.read_csv("../data/raw/payload_full.csv")

display(df.head())

print("Dataset shape:", df.shape)

print("\nAttack type distribution:")
print(df["attack_type"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

,payload,length,attack_type,label
0,c/ caridad s/n,14,norm,norm
1,"campello, el",12,norm,norm
2,40184,5,norm,norm
3,1442431887503330,16,norm,norm
4,nue37,5,norm,norm


Dataset shape: (31067, 4)

Attack type distribution:
attack_type
norm              19304
sqli              10852
xss                 532
path-traversal      290
cmdi                 89
Name: count, dtype: int64

Missing values:
payload        0
length         0
attack_type    0
label          0
dtype: int64


In [3]:
np.random.seed(42)

X = df["payload"]
y = df["attack_type"]

train_indices = []
test_indices = []

for class_name in y.unique():

    class_indices = df.index[y == class_name].to_numpy().copy()

    np.random.shuffle(class_indices)

    split_point = int(len(class_indices) * 0.80)

    train_indices.extend(class_indices[:split_point])
    test_indices.extend(class_indices[split_point:])

X_train = X.loc[train_indices]
X_test = X.loc[test_indices]

y_train = y.loc[train_indices]
y_test = y.loc[test_indices]

print("Training set size:", len(X_train))
print("Testing set size:", len(X_test))

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

Training set size: 24852
Testing set size: 6215

Training class distribution:
attack_type
norm              15443
sqli               8681
xss                 425
path-traversal      232
cmdi                 71
Name: count, dtype: int64

Testing class distribution:
attack_type
norm              3861
sqli              2171
xss                107
path-traversal      58
cmdi                18
Name: count, dtype: int64


In [5]:
from collections import Counter
import math

def generate_char_ngrams(text, ngram_range=(3, 5)):
    ngrams = []

    text = str(text)

    for n in range(ngram_range[0], ngram_range[1] + 1):
        for i in range(len(text) - n + 1):
            ngrams.append(text[i:i+n])

    return ngrams


sample_payload = "' OR 1=1 --"

sample_ngrams = generate_char_ngrams(sample_payload)

print("Original payload:", sample_payload)
print("\nGenerated character n-grams:")
print(sample_ngrams[:20])
print("\nTotal n-grams:", len(sample_ngrams))

Original payload: ' OR 1=1 --

Generated character n-grams:
["' O", ' OR', 'OR ', 'R 1', ' 1=', '1=1', '=1 ', '1 -', ' --', "' OR", ' OR ', 'OR 1', 'R 1=', ' 1=1', '1=1 ', '=1 -', '1 --', "' OR ", ' OR 1', 'OR 1=']

Total n-grams: 24


In [7]:
from collections import Counter

ngram_document_frequency = Counter()

for payload in X_train:
    unique_ngrams = set(generate_char_ngrams(payload))
    ngram_document_frequency.update(unique_ngrams)

filtered_ngrams = {
    ngram: frequency
    for ngram, frequency in ngram_document_frequency.items()
    if frequency >= 2
}

vocabulary = [
    ngram
    for ngram, frequency in Counter(filtered_ngrams).most_common(50000)
]

vocabulary_dict = {
    ngram: index
    for index, ngram in enumerate(vocabulary)
}

print("Total unique n-grams:", len(ngram_document_frequency))
print("N-grams after min_df filtering:", len(filtered_ngrams))
print("Vocabulary size:", len(vocabulary_dict))
print("\nFirst 20 vocabulary items:")
print(vocabulary[:20])

Total unique n-grams: 272216
N-grams after min_df filtering: 94367
Vocabulary size: 50000

First 20 vocabulary items:
['ele', 'sel', 'ect', 'lec', 'ct ', 'ect ', 'elec', 'lect', 'elect', 'sele', 'selec', 'lect ', 'and', ' an', ' wh', 'whe', ' whe', 'nd ', ' and', 'and ']


In [8]:
import numpy as np
from collections import Counter

document_frequency = np.zeros(len(vocabulary), dtype=np.int32)

for payload in X_train:
    unique_ngrams = set(generate_char_ngrams(payload))

    for ngram in unique_ngrams:
        if ngram in vocabulary_dict:
            document_frequency[vocabulary_dict[ngram]] += 1

n_documents = len(X_train)

idf = np.log(
    (1 + n_documents) / (1 + document_frequency)
) + 1

print("IDF shape:", idf.shape)
print("Minimum IDF:", idf.min())
print("Maximum IDF:", idf.max())

IDF shape: (50000,)
Minimum IDF: 2.4238892289299407
Maximum IDF: 9.734439387464363


In [9]:
from scipy.sparse import csr_matrix

def transform_to_tfidf(data, vocabulary_dict, idf):
    rows = []
    columns = []
    values = []

    for row_index, payload in enumerate(data):
        ngrams = generate_char_ngrams(payload)
        term_frequency = Counter(ngrams)

        total_terms = len(ngrams)

        if total_terms == 0:
            continue

        for ngram, count in term_frequency.items():
            if ngram in vocabulary_dict:
                column_index = vocabulary_dict[ngram]

                tf = count / total_terms
                tfidf_value = tf * idf[column_index]

                rows.append(row_index)
                columns.append(column_index)
                values.append(tfidf_value)

    matrix = csr_matrix(
        (values, (rows, columns)),
        shape=(len(data), len(vocabulary_dict)),
        dtype=np.float32
    )

    return matrix


X_train_tfidf = transform_to_tfidf(
    X_train, vocabulary_dict, idf
)

X_test_tfidf = transform_to_tfidf(
    X_test, vocabulary_dict, idf
)

print("Training feature matrix:", X_train_tfidf.shape)
print("Testing feature matrix:", X_test_tfidf.shape)

Training feature matrix: (24852, 50000)
Testing feature matrix: (6215, 50000)


In [10]:
class_names = sorted(y_train.unique())

class_to_index = {
    class_name: index
    for index, class_name in enumerate(class_names)
}

index_to_class = {
    index: class_name
    for class_name, index in class_to_index.items()
}

y_train_encoded = np.array([
    class_to_index[label] for label in y_train
])

y_test_encoded = np.array([
    class_to_index[label] for label in y_test
])

print("Class mapping:", class_to_index)
print("Training labels:", y_train_encoded.shape)
print("Testing labels:", y_test_encoded.shape)

Class mapping: {'cmdi': 0, 'norm': 1, 'path-traversal': 2, 'sqli': 3, 'xss': 4}
Training labels: (24852,)
Testing labels: (6215,)


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_size = X_train_tfidf.shape[1]
num_classes = len(class_names)

model = nn.Linear(input_size, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10
batch_size = 128

for epoch in range(epochs):

    model.train()

    indices = np.random.permutation(len(y_train_encoded))
    total_loss = 0

    for start in range(0, len(indices), batch_size):

        batch_indices = indices[start:start + batch_size]

        X_batch = X_train_tfidf[batch_indices].toarray().astype(np.float32)

        X_batch = torch.from_numpy(X_batch).to(device)
        y_batch = torch.tensor(
            y_train_encoded[batch_indices],
            dtype=torch.long
        ).to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(batch_indices)

    average_loss = total_loss / len(y_train_encoded)

    print(
        f"Epoch [{epoch + 1}/{epochs}], "
        f"Loss: {average_loss:.4f}"
    )

Epoch [1/10], Loss: 1.2781
Epoch [2/10], Loss: 0.8004
Epoch [3/10], Loss: 0.5298
Epoch [4/10], Loss: 0.3712
Epoch [5/10], Loss: 0.2736
Epoch [6/10], Loss: 0.2102
Epoch [7/10], Loss: 0.1669
Epoch [8/10], Loss: 0.1360
Epoch [9/10], Loss: 0.1131
Epoch [10/10], Loss: 0.0956


In [12]:
model.eval()

y_pred = []

with torch.no_grad():

    for start in range(0, len(y_test_encoded), batch_size):

        end = start + batch_size

        X_batch = X_test_tfidf[start:end].toarray().astype(np.float32)
        X_batch = torch.from_numpy(X_batch).to(device)

        outputs = model(X_batch)

        predictions = torch.argmax(outputs, dim=1)

        y_pred.extend(predictions.cpu().numpy())

y_pred = np.array(y_pred)

print("Prediction count:", len(y_pred))
print("Actual count:", len(y_test_encoded))
print("First 20 predictions:", y_pred[:20])
print("First 20 actual labels:", y_test_encoded[:20])

Prediction count: 6215
Actual count: 6215
First 20 predictions: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
First 20 actual labels: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]


In [13]:
confusion_matrix = np.zeros(
    (num_classes, num_classes),
    dtype=int
)

for actual, predicted in zip(y_test_encoded, y_pred):
    confusion_matrix[actual, predicted] += 1

print("Confusion Matrix:")
print(confusion_matrix)


accuracy = np.trace(confusion_matrix) / np.sum(confusion_matrix)

class_precision = []
class_recall = []
class_f1 = []
class_specificity = []

for i in range(num_classes):

    tp = confusion_matrix[i, i]
    fn = np.sum(confusion_matrix[i, :]) - tp
    fp = np.sum(confusion_matrix[:, i]) - tp
    tn = np.sum(confusion_matrix) - tp - fn - fp

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0
    )
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    class_precision.append(precision)
    class_recall.append(recall)
    class_f1.append(f1)
    class_specificity.append(specificity)

balanced_accuracy = np.mean(class_recall)
macro_f1 = np.mean(class_f1)

print("\n--- Overall Metrics ---")
print(f"Accuracy: {accuracy:.4f}")
print(f"Balanced Accuracy: {balanced_accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")

print("\n--- Class-Based Metrics ---")

for i, class_name in enumerate(class_names):

    print(f"\nClass: {class_name}")
    print(f"Precision: {class_precision[i]:.4f}")
    print(f"Recall: {class_recall[i]:.4f}")
    print(f"F1-score: {class_f1[i]:.4f}")
    print(f"Specificity: {class_specificity[i]:.4f}")

Confusion Matrix:
[[   8   10    0    0    0]
 [   0 3858    0    2    1]
 [   0    4   54    0    0]
 [   0    2    0 2169    0]
 [   0   14    0    0   93]]

--- Overall Metrics ---
Accuracy: 0.9947
Balanced Accuracy: 0.8486
Macro F1: 0.9000

--- Class-Based Metrics ---

Class: cmdi
Precision: 1.0000
Recall: 0.4444
F1-score: 0.6154
Specificity: 1.0000

Class: norm
Precision: 0.9923
Recall: 0.9992
F1-score: 0.9957
Specificity: 0.9873

Class: path-traversal
Precision: 1.0000
Recall: 0.9310
F1-score: 0.9643
Specificity: 1.0000

Class: sqli
Precision: 0.9991
Recall: 0.9991
F1-score: 0.9991
Specificity: 0.9995

Class: xss
Precision: 0.9894
Recall: 0.8692
F1-score: 0.9254
Specificity: 0.9998


##Logistic Regression with Focal Loss

In [14]:
class FocalLoss(nn.Module):

    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()

        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):

        log_probs = torch.log_softmax(inputs, dim=1)

        log_pt = log_probs.gather(
            1, targets.unsqueeze(1)
        ).squeeze(1)

        pt = torch.exp(log_pt)

        focal_weight = (1 - pt) ** self.gamma

        if self.alpha is not None:
            alpha_weight = self.alpha[targets]
            loss = -alpha_weight * focal_weight * log_pt
        else:
            loss = -focal_weight * log_pt

        return loss.mean()


class_counts = np.array([
    np.sum(y_train_encoded == i)
    for i in range(num_classes)
])

class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.mean()

alpha = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

print("Class counts:", class_counts)
print("Focal Loss alpha weights:", alpha.cpu().numpy())

Class counts: [   71 15443   232  8681   425]
Focal Loss alpha weights: [3.3650327  0.01547091 1.029816   0.02752187 0.5621584 ]


In [15]:
def train_focal_lr(gamma, alpha_weights, epochs=10, batch_size=128):

    model = nn.Linear(input_size, num_classes).to(device)

    criterion = FocalLoss(
        alpha=alpha_weights,
        gamma=gamma
    )

    optimizer = optim.Adam(model.parameters(), lr=0.001)

    history = []

    for epoch in range(epochs):

        model.train()

        indices = np.random.permutation(len(y_train_encoded))
        total_loss = 0

        for start in range(0, len(indices), batch_size):

            batch_indices = indices[start:start + batch_size]

            X_batch = X_train_tfidf[batch_indices].toarray().astype(np.float32)
            X_batch = torch.from_numpy(X_batch).to(device)

            y_batch = torch.tensor(
                y_train_encoded[batch_indices],
                dtype=torch.long
            ).to(device)

            optimizer.zero_grad()

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(batch_indices)

        average_loss = total_loss / len(y_train_encoded)
        history.append(average_loss)

        print(f"Gamma: {gamma} | Epoch: {epoch + 1}/{epochs} | Loss: {average_loss:.4f}")

    return model, history

In [16]:
gamma_values = [1.0, 2.0, 3.0]

focal_models = {}
focal_histories = {}

for gamma in gamma_values:

    print(f"\nTraining with gamma={gamma}")

    trained_model, history = train_focal_lr(
        gamma=gamma,
        alpha_weights=alpha
    )

    focal_models[gamma] = trained_model
    focal_histories[gamma] = history


Training with gamma=1.0
Gamma: 1.0 | Epoch: 1/10 | Loss: 0.0479
Gamma: 1.0 | Epoch: 2/10 | Loss: 0.0282
Gamma: 1.0 | Epoch: 3/10 | Loss: 0.0182
Gamma: 1.0 | Epoch: 4/10 | Loss: 0.0129
Gamma: 1.0 | Epoch: 5/10 | Loss: 0.0097
Gamma: 1.0 | Epoch: 6/10 | Loss: 0.0076
Gamma: 1.0 | Epoch: 7/10 | Loss: 0.0062
Gamma: 1.0 | Epoch: 8/10 | Loss: 0.0052
Gamma: 1.0 | Epoch: 9/10 | Loss: 0.0044
Gamma: 1.0 | Epoch: 10/10 | Loss: 0.0038

Training with gamma=2.0
Gamma: 2.0 | Epoch: 1/10 | Loss: 0.0361
Gamma: 2.0 | Epoch: 2/10 | Loss: 0.0191
Gamma: 2.0 | Epoch: 3/10 | Loss: 0.0118
Gamma: 2.0 | Epoch: 4/10 | Loss: 0.0081
Gamma: 2.0 | Epoch: 5/10 | Loss: 0.0060
Gamma: 2.0 | Epoch: 6/10 | Loss: 0.0046
Gamma: 2.0 | Epoch: 7/10 | Loss: 0.0037
Gamma: 2.0 | Epoch: 8/10 | Loss: 0.0031
Gamma: 2.0 | Epoch: 9/10 | Loss: 0.0026
Gamma: 2.0 | Epoch: 10/10 | Loss: 0.0022

Training with gamma=3.0
Gamma: 3.0 | Epoch: 1/10 | Loss: 0.0274
Gamma: 3.0 | Epoch: 2/10 | Loss: 0.0133
Gamma: 3.0 | Epoch: 3/10 | Loss: 0.0079
Gam

In [17]:
def predict_model(model, X_data, batch_size=128):

    model.eval()
    predictions = []

    with torch.no_grad():

        for start in range(0, X_data.shape[0], batch_size):

            X_batch = X_data[start:start + batch_size].toarray().astype(np.float32)
            X_batch = torch.from_numpy(X_batch).to(device)

            outputs = model(X_batch)
            batch_predictions = torch.argmax(outputs, dim=1)

            predictions.extend(batch_predictions.cpu().numpy())

    return np.array(predictions)


def calculate_metrics(y_true, y_pred):

    cm = np.zeros((num_classes, num_classes), dtype=int)

    for actual, predicted in zip(y_true, y_pred):
        cm[actual, predicted] += 1

    precision_scores = []
    recall_scores = []
    f1_scores = []
    specificity_scores = []

    for i in range(num_classes):

        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp

        precision = tp / (tp + fp) if tp + fp > 0 else 0
        recall = tp / (tp + fn) if tp + fn > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0
        specificity = tn / (tn + fp) if tn + fp > 0 else 0

        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)
        specificity_scores.append(specificity)

    return {
        "accuracy": np.trace(cm) / cm.sum(),
        "balanced_accuracy": np.mean(recall_scores),
        "macro_f1": np.mean(f1_scores),
        "macro_precision": np.mean(precision_scores),
        "macro_recall": np.mean(recall_scores),
        "macro_specificity": np.mean(specificity_scores),
        "confusion_matrix": cm,
        "class_precision": precision_scores,
        "class_recall": recall_scores,
        "class_f1": f1_scores,
        "class_specificity": specificity_scores
    }


focal_results = {}

for gamma, trained_model in focal_models.items():

    predictions = predict_model(trained_model, X_test_tfidf)

    metrics = calculate_metrics(y_test_encoded, predictions)

    focal_results[gamma] = metrics

    print(f"\nGamma: {gamma}")
    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print(f"Balanced Accuracy: {metrics['balanced_accuracy']:.4f}")
    print(f"Macro F1: {metrics['macro_f1']:.4f}")
    print(f"Macro Specificity: {metrics['macro_specificity']:.4f}")


Gamma: 1.0
Accuracy: 0.9974
Balanced Accuracy: 0.9659
Macro F1: 0.9703
Macro Specificity: 0.9991

Gamma: 2.0
Accuracy: 0.9823
Balanced Accuracy: 0.9721
Macro F1: 0.8399
Macro Specificity: 0.9962

Gamma: 3.0
Accuracy: 0.9823
Balanced Accuracy: 0.9721
Macro F1: 0.8399
Macro Specificity: 0.9962


In [19]:
for gamma, metrics in focal_results.items():

    print(f"\nGamma: {gamma}")

    for i, class_name in enumerate(class_names):

        print(
            f"{class_name}: "
            f"Precision={metrics['class_precision'][i]:.4f}, "
            f"Recall={metrics['class_recall'][i]:.4f}, "
            f"F1={metrics['class_f1'][i]:.4f}, "
            f"Specificity={metrics['class_specificity'][i]:.4f}"
        )

    print("Confusion Matrix:")
    print(metrics["confusion_matrix"])


Gamma: 1.0
cmdi: Precision=0.9375, Recall=0.8333, F1=0.8824, Specificity=0.9998
norm: Precision=0.9987, Recall=0.9972, F1=0.9979, Specificity=0.9979
path-traversal: Precision=1.0000, Recall=1.0000, F1=1.0000, Specificity=1.0000
sqli: Precision=0.9982, Recall=0.9991, F1=0.9986, Specificity=0.9990
xss: Precision=0.9469, Recall=1.0000, F1=0.9727, Specificity=0.9990
Confusion Matrix:
[[  15    3    0    0    0]
 [   1 3850    0    4    6]
 [   0    0   58    0    0]
 [   0    2    0 2169    0]
 [   0    0    0    0  107]]

Gamma: 2.0
cmdi: Precision=0.1404, Recall=0.8889, F1=0.2424, Specificity=0.9842
norm: Precision=0.9992, Recall=0.9725, F1=0.9857, Specificity=0.9987
path-traversal: Precision=1.0000, Recall=1.0000, F1=1.0000, Specificity=1.0000
sqli: Precision=0.9986, Recall=0.9991, F1=0.9988, Specificity=0.9993
xss: Precision=0.9469, Recall=1.0000, F1=0.9727, Specificity=0.9990
Confusion Matrix:
[[  16    2    0    0    0]
 [  97 3755    0    3    6]
 [   0    0   58    0    0]
 [   1 

In [20]:
import os
import pandas as pd

os.makedirs("../reports", exist_ok=True)

summary_rows = []

summary_rows.append({
    "Model": "Logistic Regression",
    "Loss Function": "CrossEntropy",
    "Gamma": "-",
    "Accuracy": 0.9947,
    "Balanced Accuracy": 0.8486,
    "Macro F1": 0.9000,
    "Macro Specificity": "-"
})

for gamma, metrics in focal_results.items():

    summary_rows.append({
        "Model": "Logistic Regression",
        "Loss Function": "Class-Weighted Focal Loss",
        "Gamma": gamma,
        "Accuracy": metrics["accuracy"],
        "Balanced Accuracy": metrics["balanced_accuracy"],
        "Macro F1": metrics["macro_f1"],
        "Macro Specificity": metrics["macro_specificity"]
    })

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    "../reports/logistic_regression_results.csv",
    index=False,
    float_format="%.4f"
)

class_rows = []

for gamma, metrics in focal_results.items():

    for i, class_name in enumerate(class_names):

        class_rows.append({
            "Gamma": gamma,
            "Class": class_name,
            "Precision": metrics["class_precision"][i],
            "Recall": metrics["class_recall"][i],
            "F1 Score": metrics["class_f1"][i],
            "Specificity": metrics["class_specificity"][i]
        })

class_df = pd.DataFrame(class_rows)

class_df.to_csv(
    "../reports/focal_loss_class_metrics.csv",
    index=False,
    float_format="%.4f"
)

for gamma, metrics in focal_results.items():

    confusion_df = pd.DataFrame(
        metrics["confusion_matrix"],
        index=class_names,
        columns=class_names
    )

    confusion_df.index.name = "Actual / Predicted"

    confusion_df.to_csv(
        f"../reports/focal_loss_confusion_matrix_gamma_{gamma}.csv"
    )



print("\nOverall Results:")
display(summary_df)

print("\nClass-Based Results:")
display(class_df)


Overall Results:


,Model,Loss Function,Gamma,Accuracy,Balanced Accuracy,Macro F1,Macro Specificity
0,Logistic Regression,CrossEntropy,-,0.994700,0.848600,0.900000,-
1,Logistic Regression,Class-Weighted Focal Loss,1.0,0.997426,0.965913,0.970325,0.999149
2,Logistic Regression,Class-Weighted Focal Loss,2.0,0.982301,0.972103,0.839939,0.996237
3,Logistic Regression,Class-Weighted Focal Loss,3.0,0.982301,0.972103,0.839939,0.996237



Class-Based Results:


,Gamma,Class,Precision,Recall,F1 Score,Specificity
0,1.0,cmdi,0.937500,0.833333,0.882353,0.999839
1,1.0,norm,0.998703,0.997151,0.997926,0.997876
2,1.0,path-traversal,1.000000,1.000000,1.000000,1.000000
3,1.0,sqli,0.998159,0.999079,0.998619,0.999011
4,1.0,xss,0.946903,1.000000,0.972727,0.999018
5,2.0,cmdi,0.140351,0.888889,0.242424,0.984186
6,2.0,norm,0.999202,0.972546,0.985694,0.998726
7,2.0,path-traversal,1.000000,1.000000,1.000000,1.000000
8,2.0,sqli,0.998619,0.999079,0.998849,0.999258
9,2.0,xss,0.946903,1.000000,0.972727,0.999018


In [21]:
from scipy.sparse import save_npz
import os

os.makedirs("../data/processed", exist_ok=True)

save_npz("../data/processed/X_train_tfidf.npz", X_train_tfidf)
save_npz("../data/processed/X_test_tfidf.npz", X_test_tfidf)

np.save("../data/processed/y_train_encoded.npy", y_train_encoded)
np.save("../data/processed/y_test_encoded.npy", y_test_encoded)
np.save("../data/processed/class_names.npy", np.array(class_names))



##Random Oversampling

In [22]:
np.random.seed(42)

class_counts = np.bincount(y_train_encoded, minlength=num_classes)
target_count = class_counts.max()

oversampled_indices = []

for class_id in range(num_classes):
    class_indices = np.where(y_train_encoded == class_id)[0]

    if len(class_indices) < target_count:
        additional_indices = np.random.choice(
            class_indices,
            size=target_count - len(class_indices),
            replace=True
        )

        class_indices = np.concatenate([class_indices, additional_indices])

    oversampled_indices.extend(class_indices)

oversampled_indices = np.array(oversampled_indices)
np.random.shuffle(oversampled_indices)

X_train_ros = X_train_tfidf[oversampled_indices]
y_train_ros = y_train_encoded[oversampled_indices]

print("Original training shape:", X_train_tfidf.shape)
print("Oversampled training shape:", X_train_ros.shape)
print("Test shape:", X_test_tfidf.shape)

print("\nClass distribution after oversampling:")

for class_id, class_name in enumerate(class_names):
    count = np.sum(y_train_ros == class_id)
    print(f"{class_name}: {count}")

Original training shape: (24852, 50000)
Oversampled training shape: (77215, 50000)
Test shape: (6215, 50000)

Class distribution after oversampling:
cmdi: 15443
norm: 15443
path-traversal: 15443
sqli: 15443
xss: 15443


In [23]:
ros_lr_model = nn.Linear(input_size, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(ros_lr_model.parameters(), lr=0.001)

epochs = 10
batch_size = 128

ros_losses = []

for epoch in range(epochs):
    ros_lr_model.train()
    epoch_loss = 0

    permutation = np.random.permutation(len(y_train_ros))

    for i in range(0, len(permutation), batch_size):
        indices = permutation[i:i + batch_size]

        batch_x = X_train_ros[indices].toarray().astype(np.float32)
        batch_y = y_train_ros[indices]

        inputs = torch.tensor(batch_x, dtype=torch.float32).to(device)
        targets = torch.tensor(batch_y, dtype=torch.long).to(device)

        optimizer.zero_grad()

        outputs = ros_lr_model(inputs)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(indices)

    average_loss = epoch_loss / len(y_train_ros)
    ros_losses.append(average_loss)

    print(f"Epoch {epoch + 1}/{epochs}, Loss: {average_loss:.4f}")

Epoch 1/10, Loss: 0.7261
Epoch 2/10, Loss: 0.2246
Epoch 3/10, Loss: 0.1161
Epoch 4/10, Loss: 0.0730
Epoch 5/10, Loss: 0.0511
Epoch 6/10, Loss: 0.0383
Epoch 7/10, Loss: 0.0301
Epoch 8/10, Loss: 0.0246
Epoch 9/10, Loss: 0.0207
Epoch 10/10, Loss: 0.0178


In [24]:
ros_lr_model.eval()

ros_predictions = []

with torch.no_grad():
    for i in range(0, len(y_test_encoded), 128):
        batch_x = X_test_tfidf[i:i + 128].toarray().astype(np.float32)

        inputs = torch.tensor(batch_x, dtype=torch.float32).to(device)

        outputs = ros_lr_model(inputs)
        predictions = torch.argmax(outputs, dim=1)

        ros_predictions.extend(predictions.cpu().numpy())

ros_predictions = np.array(ros_predictions)

print("Prediction count:", len(ros_predictions))
print("Test sample count:", len(y_test_encoded))

Prediction count: 6215
Test sample count: 6215


In [25]:
ros_results = calculate_metrics(
    y_test_encoded,
    ros_predictions
)

print("Returned values:", len(ros_results))

for index, result in enumerate(ros_results):
    print(f"\nResult {index}:")
    print(result)

Returned values: 11

Result 0:
accuracy

Result 1:
balanced_accuracy

Result 2:
macro_f1

Result 3:
macro_precision

Result 4:
macro_recall

Result 5:
macro_specificity

Result 6:
confusion_matrix

Result 7:
class_precision

Result 8:
class_recall

Result 9:
class_f1

Result 10:
class_specificity


In [26]:
ros_results = calculate_metrics(
    y_test_encoded,
    ros_predictions
)

print("Random Oversampling - Logistic Regression")
print("------------------------------------------")

print(f"Accuracy: {ros_results['accuracy']:.4f}")
print(f"Balanced Accuracy: {ros_results['balanced_accuracy']:.4f}")
print(f"Macro F1: {ros_results['macro_f1']:.4f}")
print(f"Macro Precision: {ros_results['macro_precision']:.4f}")
print(f"Macro Recall: {ros_results['macro_recall']:.4f}")
print(f"Macro Specificity: {ros_results['macro_specificity']:.4f}")

print("\nClass-based Metrics:")

for i, class_name in enumerate(class_names):
    print(
        f"{class_name}: "
        f"Precision={ros_results['class_precision'][i]:.4f}, "
        f"Recall={ros_results['class_recall'][i]:.4f}, "
        f"F1={ros_results['class_f1'][i]:.4f}, "
        f"Specificity={ros_results['class_specificity'][i]:.4f}"
    )

print("\nConfusion Matrix:")
print(ros_results['confusion_matrix'])

Random Oversampling - Logistic Regression
------------------------------------------
Accuracy: 0.9973
Balanced Accuracy: 0.9659
Macro F1: 0.9628
Macro Precision: 0.9604
Macro Recall: 0.9659
Macro Specificity: 0.9992

Class-based Metrics:
cmdi: Precision=0.8824, Recall=0.8333, F1=0.8571, Specificity=0.9997
norm: Precision=0.9987, Recall=0.9969, F1=0.9978, Specificity=0.9979
path-traversal: Precision=0.9831, Recall=1.0000, F1=0.9915, Specificity=0.9998
sqli: Precision=0.9991, Recall=0.9991, F1=0.9991, Specificity=0.9995
xss: Precision=0.9386, Recall=1.0000, F1=0.9683, Specificity=0.9989

Confusion Matrix:
[[  15    3    0    0    0]
 [   2 3849    1    2    7]
 [   0    0   58    0    0]
 [   0    2    0 2169    0]
 [   0    0    0    0  107]]


In [27]:
import pandas as pd
import os

os.makedirs("../reports", exist_ok=True)

ros_rows = []

ros_rows.append({
    "model": "Logistic Regression",
    "method": "Random Oversampling",
    "class": "overall",
    "accuracy": ros_results["accuracy"],
    "balanced_accuracy": ros_results["balanced_accuracy"],
    "macro_f1": ros_results["macro_f1"],
    "macro_precision": ros_results["macro_precision"],
    "macro_recall": ros_results["macro_recall"],
    "macro_specificity": ros_results["macro_specificity"]
})

for i, class_name in enumerate(class_names):
    ros_rows.append({
        "model": "Logistic Regression",
        "method": "Random Oversampling",
        "class": class_name,
        "precision": ros_results["class_precision"][i],
        "recall": ros_results["class_recall"][i],
        "f1": ros_results["class_f1"][i],
        "specificity": ros_results["class_specificity"][i]
    })

ros_df = pd.DataFrame(ros_rows)

ros_df.to_csv(
    "../reports/logistic_regression_random_oversampling.csv",
    index=False
)



##SMOTE

In [28]:
import numpy as np
from scipy.sparse import csr_matrix, vstack

np.random.seed(42)

class_counts = np.bincount(y_train_encoded, minlength=num_classes)
target_count = class_counts.max()

print("Original class distribution:")

for class_id, class_name in enumerate(class_names):
    print(f"{class_name}: {class_counts[class_id]}")

print("\nTarget count:", target_count)

Original class distribution:
cmdi: 71
norm: 15443
path-traversal: 232
sqli: 8681
xss: 425

Target count: 15443


In [29]:
from scipy.sparse import vstack

def generate_smote_samples(X, y, target_count, k_neighbors=5):
    synthetic_features = []
    synthetic_labels = []

    for class_id in range(num_classes):
        class_indices = np.where(y == class_id)[0]
        class_count = len(class_indices)

        if class_count >= target_count:
            continue

        class_samples = X[class_indices]

        class_dense = class_samples.toarray().astype(np.float32)

        distances = np.sum(
            (class_dense[:, None, :] - class_dense[None, :, :]) ** 2,
            axis=2
        )

        neighbor_indices = np.argsort(distances, axis=1)[:, 1:k_neighbors + 1]

        required_samples = target_count - class_count
        generated_samples = []

        for _ in range(required_samples):
            sample_index = np.random.randint(class_count)
            neighbor_index = np.random.choice(neighbor_indices[sample_index])

            sample = class_dense[sample_index]
            neighbor = class_dense[neighbor_index]

            interpolation = np.random.random()

            synthetic_sample = sample + interpolation * (neighbor - sample)

            generated_samples.append(synthetic_sample)

        synthetic_features.append(
            csr_matrix(np.array(generated_samples))
        )

        synthetic_labels.extend([class_id] * required_samples)

    if synthetic_features:
        X_synthetic = vstack(synthetic_features, format="csr")
        y_synthetic = np.array(synthetic_labels, dtype=np.int64)

        X_resampled = vstack([X, X_synthetic], format="csr")
        y_resampled = np.concatenate([y, y_synthetic])
    else:
        X_resampled = X.copy()
        y_resampled = y.copy()

    return X_resampled, y_resampled

In [30]:
def generate_smote_samples(X, y, target_count, k_neighbors=5):
    synthetic_features = []
    synthetic_labels = []

    for class_id in range(num_classes):
        class_indices = np.where(y == class_id)[0]
        class_count = len(class_indices)

        if class_count >= target_count:
            continue

        class_samples = X[class_indices]

        similarity = (class_samples @ class_samples.T).toarray()
        np.fill_diagonal(similarity, -np.inf)

        neighbor_indices = np.argsort(
            similarity,
            axis=1
        )[:, -min(k_neighbors, class_count - 1):]

        required_samples = target_count - class_count

        for _ in range(required_samples):
            sample_index = np.random.randint(class_count)
            neighbor_index = np.random.choice(
                neighbor_indices[sample_index]
            )

            sample = class_samples[sample_index]
            neighbor = class_samples[neighbor_index]

            interpolation = np.random.random()

            synthetic_sample = (
                sample + interpolation * (neighbor - sample)
            )

            synthetic_features.append(synthetic_sample)
            synthetic_labels.append(class_id)

    X_synthetic = vstack(synthetic_features, format="csr")
    y_synthetic = np.array(synthetic_labels, dtype=np.int64)

    X_resampled = vstack([X, X_synthetic], format="csr")
    y_resampled = np.concatenate([y, y_synthetic])

    return X_resampled, y_resampled

In [31]:
X_train_smote, y_train_smote = generate_smote_samples(
    X_train_tfidf,
    y_train_encoded,
    target_count,
    k_neighbors=5
)

print("Original training shape:", X_train_tfidf.shape)
print("SMOTE training shape:", X_train_smote.shape)
print("Test shape:", X_test_tfidf.shape)

print("\nClass distribution after SMOTE:")

for class_id, class_name in enumerate(class_names):
    count = np.sum(y_train_smote == class_id)
    print(f"{class_name}: {count}")

Original training shape: (24852, 50000)
SMOTE training shape: (77215, 50000)
Test shape: (6215, 50000)

Class distribution after SMOTE:
cmdi: 15443
norm: 15443
path-traversal: 15443
sqli: 15443
xss: 15443


In [32]:
smote_lr_model = nn.Linear(input_size, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(smote_lr_model.parameters(), lr=0.001)

epochs = 10
batch_size = 128

smote_losses = []

for epoch in range(epochs):
    smote_lr_model.train()
    epoch_loss = 0

    permutation = np.random.permutation(len(y_train_smote))

    for i in range(0, len(permutation), batch_size):
        indices = permutation[i:i + batch_size]

        batch_x = X_train_smote[indices].toarray().astype(np.float32)
        batch_y = y_train_smote[indices]

        inputs = torch.tensor(batch_x, dtype=torch.float32).to(device)
        targets = torch.tensor(batch_y, dtype=torch.long).to(device)

        optimizer.zero_grad()

        outputs = smote_lr_model(inputs)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(indices)

    average_loss = epoch_loss / len(y_train_smote)
    smote_losses.append(average_loss)

    print(f"Epoch {epoch + 1}/{epochs}, Loss: {average_loss:.4f}")

Epoch 1/10, Loss: 0.6957
Epoch 2/10, Loss: 0.1999
Epoch 3/10, Loss: 0.0983
Epoch 4/10, Loss: 0.0591
Epoch 5/10, Loss: 0.0395
Epoch 6/10, Loss: 0.0282
Epoch 7/10, Loss: 0.0210
Epoch 8/10, Loss: 0.0162
Epoch 9/10, Loss: 0.0128
Epoch 10/10, Loss: 0.0104


In [33]:
smote_lr_model.eval()

smote_predictions = []

with torch.no_grad():
    for i in range(0, len(y_test_encoded), 128):
        batch_x = X_test_tfidf[i:i + 128].toarray().astype(np.float32)

        inputs = torch.tensor(batch_x, dtype=torch.float32).to(device)

        outputs = smote_lr_model(inputs)
        predictions = torch.argmax(outputs, dim=1)

        smote_predictions.extend(predictions.cpu().numpy())

smote_predictions = np.array(smote_predictions)

print("Prediction count:", len(smote_predictions))
print("Test sample count:", len(y_test_encoded))

Prediction count: 6215
Test sample count: 6215


In [34]:
smote_results = calculate_metrics(
    y_test_encoded,
    smote_predictions
)

print("SMOTE - Logistic Regression")
print("-----------------------------------")

print(f"Accuracy: {smote_results['accuracy']:.4f}")
print(f"Balanced Accuracy: {smote_results['balanced_accuracy']:.4f}")
print(f"Macro F1: {smote_results['macro_f1']:.4f}")
print(f"Macro Precision: {smote_results['macro_precision']:.4f}")
print(f"Macro Recall: {smote_results['macro_recall']:.4f}")
print(f"Macro Specificity: {smote_results['macro_specificity']:.4f}")

print("\nClass-based Metrics:")

for i, class_name in enumerate(class_names):
    print(
        f"{class_name}: "
        f"Precision={smote_results['class_precision'][i]:.4f}, "
        f"Recall={smote_results['class_recall'][i]:.4f}, "
        f"F1={smote_results['class_f1'][i]:.4f}, "
        f"Specificity={smote_results['class_specificity'][i]:.4f}"
    )

print("\nConfusion Matrix:")
print(smote_results['confusion_matrix'])

SMOTE - Logistic Regression
-----------------------------------
Accuracy: 0.9979
Balanced Accuracy: 0.9661
Macro F1: 0.9706
Macro Precision: 0.9765
Macro Recall: 0.9661
Macro Specificity: 0.9993

Class-based Metrics:
cmdi: Precision=0.9375, Recall=0.8333, F1=0.8824, Specificity=0.9998
norm: Precision=0.9987, Recall=0.9979, F1=0.9983, Specificity=0.9979
path-traversal: Precision=0.9831, Recall=1.0000, F1=0.9915, Specificity=0.9998
sqli: Precision=0.9991, Recall=0.9991, F1=0.9991, Specificity=0.9995
xss: Precision=0.9640, Recall=1.0000, F1=0.9817, Specificity=0.9993

Confusion Matrix:
[[  15    3    0    0    0]
 [   1 3853    1    2    4]
 [   0    0   58    0    0]
 [   0    2    0 2169    0]
 [   0    0    0    0  107]]


In [35]:
smote_rows = []

smote_rows.append({
    "model": "Logistic Regression",
    "method": "SMOTE",
    "class": "overall",
    "accuracy": smote_results["accuracy"],
    "balanced_accuracy": smote_results["balanced_accuracy"],
    "macro_f1": smote_results["macro_f1"],
    "macro_precision": smote_results["macro_precision"],
    "macro_recall": smote_results["macro_recall"],
    "macro_specificity": smote_results["macro_specificity"]
})

for i, class_name in enumerate(class_names):
    smote_rows.append({
        "model": "Logistic Regression",
        "method": "SMOTE",
        "class": class_name,
        "precision": smote_results["class_precision"][i],
        "recall": smote_results["class_recall"][i],
        "f1": smote_results["class_f1"][i],
        "specificity": smote_results["class_specificity"][i]
    })

smote_df = pd.DataFrame(smote_rows)

smote_df.to_csv(
    "../reports/logistic_regression_smote.csv",
    index=False
)



##Borderline-SMOTE

In [36]:
def find_danger_samples(X, y, m_neighbors=10):
    danger_samples = {}

    row_norms = np.sqrt(X.multiply(X).sum(axis=1)).A1
    minority_classes = np.where(
        np.bincount(y, minlength=num_classes) < np.bincount(y, minlength=num_classes).max()
    )[0]

    for class_id in minority_classes:
        class_indices = np.where(y == class_id)[0]
        danger_indices = []

        for sample_index in class_indices:
            similarities = (X[sample_index] @ X.T).toarray().ravel()
            denominators = row_norms[sample_index] * row_norms
            similarities = np.divide(
                similarities,
                denominators,
                out=np.zeros_like(similarities),
                where=denominators != 0
            )

            similarities[sample_index] = -np.inf

            neighbor_count = min(m_neighbors, len(y) - 1)
            neighbor_indices = np.argpartition(
                similarities,
                -neighbor_count
            )[-neighbor_count:]

            other_class_count = np.sum(y[neighbor_indices] != class_id)

            if neighbor_count / 2 <= other_class_count < neighbor_count:
                danger_indices.append(sample_index)

        danger_samples[class_id] = np.array(danger_indices)

    return danger_samples


danger_samples = find_danger_samples(
    X_train_tfidf,
    y_train_encoded,
    m_neighbors=10
)

for class_id, indices in danger_samples.items():
    print(f"{class_names[class_id]}: {len(indices)} danger samples")

cmdi: 19 danger samples
path-traversal: 10 danger samples
sqli: 335 danger samples
xss: 6 danger samples


In [37]:
def generate_borderline_smote(X, y, danger_samples, target_count, k_neighbors=5):
    synthetic_features = []
    synthetic_labels = []

    for class_id, danger_indices in danger_samples.items():
        class_indices = np.where(y == class_id)[0]
        class_count = len(class_indices)

        if class_count >= target_count:
            continue

        class_samples = X[class_indices]
        similarity = (class_samples @ class_samples.T).toarray()

        row_norms = np.sqrt(class_samples.multiply(class_samples).sum(axis=1)).A1
        denominators = row_norms[:, None] * row_norms[None, :]

        similarity = np.divide(
            similarity,
            denominators,
            out=np.zeros_like(similarity),
            where=denominators != 0
        )

        np.fill_diagonal(similarity, -np.inf)

        neighbor_count = min(k_neighbors, class_count - 1)
        neighbor_indices = np.argsort(
            similarity,
            axis=1
        )[:, -neighbor_count:]

        if len(danger_indices) == 0:
            danger_indices = class_indices

        danger_local_indices = np.searchsorted(class_indices, danger_indices)
        required_samples = target_count - class_count

        for _ in range(required_samples):
            sample_local_index = np.random.choice(danger_local_indices)
            neighbor_local_index = np.random.choice(
                neighbor_indices[sample_local_index]
            )

            sample = class_samples[sample_local_index]
            neighbor = class_samples[neighbor_local_index]

            interpolation = np.random.random()

            synthetic_sample = (
                sample + interpolation * (neighbor - sample)
            )

            synthetic_features.append(synthetic_sample)
            synthetic_labels.append(class_id)

    X_synthetic = vstack(synthetic_features, format="csr")
    y_synthetic = np.array(synthetic_labels, dtype=np.int64)

    X_resampled = vstack([X, X_synthetic], format="csr")
    y_resampled = np.concatenate([y, y_synthetic])

    return X_resampled, y_resampled


np.random.seed(42)

X_train_borderline, y_train_borderline = generate_borderline_smote(
    X_train_tfidf,
    y_train_encoded,
    danger_samples,
    target_count,
    k_neighbors=5
)

print("Original shape:", X_train_tfidf.shape)
print("Borderline-SMOTE shape:", X_train_borderline.shape)
print("Class distribution:", np.bincount(y_train_borderline))

Original shape: (24852, 50000)
Borderline-SMOTE shape: (77215, 50000)
Class distribution: [15443 15443 15443 15443 15443]


In [38]:
borderline_lr_model = nn.Linear(input_size, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(borderline_lr_model.parameters(), lr=0.001)

epochs = 10
batch_size = 128

for epoch in range(epochs):
    borderline_lr_model.train()
    epoch_loss = 0

    indices = np.random.permutation(len(y_train_borderline))

    for start in range(0, len(indices), batch_size):
        batch_indices = indices[start:start + batch_size]

        X_batch = X_train_borderline[batch_indices].toarray().astype(np.float32)
        y_batch = y_train_borderline[batch_indices]

        X_batch = torch.tensor(X_batch, dtype=torch.float32, device=device)
        y_batch = torch.tensor(y_batch, dtype=torch.long, device=device)

        optimizer.zero_grad()

        outputs = borderline_lr_model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch + 1}/{epochs}, Loss: {epoch_loss:.4f}")

Epoch 1/10, Loss: 469.2866
Epoch 2/10, Loss: 145.7915
Epoch 3/10, Loss: 70.3513
Epoch 4/10, Loss: 40.9576
Epoch 5/10, Loss: 26.4200
Epoch 6/10, Loss: 18.1473
Epoch 7/10, Loss: 13.0126
Epoch 8/10, Loss: 9.6318
Epoch 9/10, Loss: 7.2986
Epoch 10/10, Loss: 5.6420


In [39]:
borderline_lr_model.eval()

borderline_predictions = []

with torch.no_grad():
    for start in range(0, X_test_tfidf.shape[0], 128):
        X_batch = X_test_tfidf[start:start + 128].toarray().astype(np.float32)
        X_batch = torch.tensor(X_batch, dtype=torch.float32, device=device)

        outputs = borderline_lr_model(X_batch)
        predictions = torch.argmax(outputs, dim=1)

        borderline_predictions.extend(predictions.cpu().numpy())

borderline_predictions = np.array(borderline_predictions)

print("Prediction count:", len(borderline_predictions))

Prediction count: 6215


In [40]:
borderline_results = calculate_metrics(
    y_test_encoded,
    borderline_predictions
)

print("Accuracy:", borderline_results["accuracy"])
print("Balanced Accuracy:", borderline_results["balanced_accuracy"])
print("Macro F1:", borderline_results["macro_f1"])
print("Macro Precision:", borderline_results["macro_precision"])
print("Macro Recall:", borderline_results["macro_recall"])
print("Macro Specificity:", borderline_results["macro_specificity"])

for i, class_name in enumerate(class_names):
    print(
        class_name,
        "Precision:", borderline_results["class_precision"][i],
        "Recall:", borderline_results["class_recall"][i],
        "F1:", borderline_results["class_f1"][i],
        "Specificity:", borderline_results["class_specificity"][i]
    )

Accuracy: 0.9964601769911504
Balanced Accuracy: 0.9418209960489886
Macro F1: 0.949412119811187
Macro Precision: 0.9585333268045269
Macro Recall: 0.9418209960489886
Macro Specificity: 0.9986213175796103
cmdi Precision: 0.875 Recall: 0.7777777777777778 F1: 0.823529411764706 Specificity: 0.999677263191867
norm Precision: 0.9968944099378882 Recall: 0.9976689976689976 F1: 0.9972815533980584 Specificity: 0.9949022939677146
path-traversal Precision: 0.9508196721311475 Recall: 1.0 F1: 0.9747899159663865 Specificity: 0.9995127497157706
sqli Precision: 0.9990787655458314 Recall: 0.9990787655458314 F1: 0.9990787655458314 Specificity: 0.9995054401582592
xss Precision: 0.970873786407767 Recall: 0.9345794392523364 F1: 0.9523809523809522 Specificity: 0.99950884086444


In [41]:
borderline_rows = []

borderline_rows.append({
    "model": "Logistic Regression",
    "method": "Borderline-SMOTE",
    "class": "overall",
    "accuracy": borderline_results["accuracy"],
    "balanced_accuracy": borderline_results["balanced_accuracy"],
    "macro_f1": borderline_results["macro_f1"],
    "macro_precision": borderline_results["macro_precision"],
    "macro_recall": borderline_results["macro_recall"],
    "macro_specificity": borderline_results["macro_specificity"]
})

for i, class_name in enumerate(class_names):
    borderline_rows.append({
        "model": "Logistic Regression",
        "method": "Borderline-SMOTE",
        "class": class_name,
        "precision": borderline_results["class_precision"][i],
        "recall": borderline_results["class_recall"][i],
        "f1": borderline_results["class_f1"][i],
        "specificity": borderline_results["class_specificity"][i]
    })

borderline_df = pd.DataFrame(borderline_rows)

borderline_df.to_csv(
    "../reports/logistic_regression_borderline_smote.csv",
    index=False
)

